# Data Verification Pipeline

This pipeline is designed for verifying and correcting 2D eye data alignment.
Use this after running the synchronization pipeline and before Kerr degree conversion.

In [1]:
# Imports
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
from eye_tracking_system_tools.preprocessing import BlockSync, utility_functions as uf


In [2]:
def load_eye_data(block):
    """
    Load the eye dataframes from CSV files created by the synchronization pipeline.
    No rotation matrices are loaded as rotation is no longer used.
    :param block: The current blocksync class
    :return: None
    """
    try:
        block.left_eye_data = pd.read_csv(block.analysis_path / 'left_eye_data.csv', index_col=0, engine='python')
        block.right_eye_data = pd.read_csv(block.analysis_path / 'right_eye_data.csv', index_col=0, engine='python')
        print(f'Loaded eye data for block {block.block_num}')
    except FileNotFoundError:
        print('Eye data files not found. Run the synchronization pipeline first!')
        raise


# Helper functions for data correction (no rotation needed)
def horizontal_flip_eye_data(df: pd.DataFrame, frame_width: int) -> pd.DataFrame:
    df2 = df.copy()
    df2['center_x'] = frame_width - df2['center_x']
    df2['phi'] = (180 - df2['phi']) % 360
    return df2


def rotate_phi_only(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()
    df2['phi'] = (df2['phi'] + 90) % 360
    return df2


def flip_x_only(df: pd.DataFrame, frame_width: int) -> pd.DataFrame:
    df2 = df.copy()
    df2['center_x'] = frame_width - df2['center_x']
    return df2


def interactive_eye_data_corrector_synced(block, eye, ref_point_xy=None):
    """
    Interactive synchronized video + ellipse editor with Play/Pause, correction, Save,
    Flip-Dot, and Skip-forward/backward (1 minute) buttons.

    Parameters
    ----------
    block : BlockSync
        Your BlockSync instance with loaded eye_data and rotation_matrix attributes.
    eye : str
        'left' or 'right'
    ref_point_xy : tuple[int,int] or None
        If provided, a (x,y) coordinate in raw frame space to draw as a blue dot on every frame.
    """
    import cv2
    import numpy as np
    import pandas as pd

    # 1) select data & video
    if eye.lower() == 'left':
        df_orig = block.left_eye_data.copy()
        rot_mat = np.array(block.left_rotation_matrix, dtype=np.float32)
        rot_angle = float(block.left_rotation_angle)
        video = block.le_videos[0]
    else:
        df_orig = block.right_eye_data.copy()
        rot_mat = np.array(block.right_rotation_matrix, dtype=np.float32)
        rot_angle = float(block.right_rotation_angle)
        video = block.re_videos[0]

    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {eye} video: {video}")

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    skip_frames = int(fps * 60)  # skip 1 minute

    # 2) prepare DataFrame & frame index column
    df_current = df_orig.copy()
    frame_col = 'eye_frame' if 'eye_frame' in df_current.columns else 'frame'

    # 3) define buttons & layout
    buttons = {
        'Play': ((10, 10), (180, 60)),
        'Pause': ((10, 80), (180, 130)),
        'Un-rotate': ((10, 150), (180, 200)),
        'X-flip': ((10, 220), (180, 270)),
        'Re-rotate': ((10, 290), (180, 340)),
        'Phi+90': ((10, 360), (180, 410)),
        'FlipX-only': ((10, 430), (180, 480)),
        'Flip Dot': ((10, 500), (180, 550)),
        'Bwd': ((10, 570), (180, 620)),
        'Fwd': ((10, 640), (180, 690)),
        'Save': ((10, 710), (180, 760)),
        'Quit': ((10, 780), (180, 830)),
    }
    ctrl_h, ctrl_w = 860, 200

    def draw_controls():
        img = np.zeros((ctrl_h, ctrl_w, 3), dtype=np.uint8)
        for name, ((x1, y1), (x2, y2)) in buttons.items():
            cv2.rectangle(img, (x1, y1), (x2, y2), (50, 50, 50), -1)
            cv2.rectangle(img, (x1, y1), (x2, y2), (200, 200, 200), 2)
            cv2.putText(img, name, (x1 + 5, y1 + 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2, cv2.LINE_AA)
        return img

    controls_img = draw_controls()
    cv2.namedWindow('Controls', cv2.WINDOW_NORMAL)

    # 4) interaction state
    running = True
    playing = False
    current_ref = ref_point_xy
    last_frame = None

    # 5) mouse callback
    def on_mouse(event, x, y, flags, param):
        nonlocal df_current, running, playing, current_ref, last_frame
        if event != cv2.EVENT_LBUTTONDOWN:
            return
        for name, ((x1, y1), (x2, y2)) in buttons.items():
            if x1 <= x <= x2 and y1 <= y <= y2:
                if name == 'Play':
                    playing = True
                elif name == 'Pause':
                    playing = False
                elif name == 'Un-rotate':
                    df_current = apply_inverse_rotation(df_current, rot_mat, rot_angle)
                elif name == 'X-flip':
                    df_current = horizontal_flip_eye_data(df_current, W)
                elif name == 'Re-rotate':
                    df_current = apply_rotation(df_current, rot_mat, rot_angle)
                elif name == 'Phi+90':
                    df_current = rotate_phi_only(df_current)
                elif name == 'FlipX-only':
                    df_current = flip_x_only(df_current, W)
                elif name == 'Flip Dot' and current_ref is not None:
                    x0, y0 = current_ref
                    current_ref = (W - x0, y0)
                elif name == 'Bwd':
                    # skip backward 1 minute
                    idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1
                    new_idx = max(idx - skip_frames, 0)
                    cap.set(cv2.CAP_PROP_POS_FRAMES, new_idx)
                    last_frame = None
                elif name == 'Fwd':
                    # skip forward 1 minute
                    idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1
                    new_idx = min(idx + skip_frames, total_frames - 1)
                    cap.set(cv2.CAP_PROP_POS_FRAMES, new_idx)
                    last_frame = None
                elif name == 'Save':
                    if eye.lower() == 'left':
                        block.left_eye_data = df_current.copy()
                    else:
                        block.right_eye_data = df_current.copy()
                    print(f"{eye.capitalize()} eye data saved.")
                elif name == 'Quit':
                    running = False
                break

    cv2.setMouseCallback('Controls', on_mouse)

    # 6) play/pause loop
    while running:
        if playing or last_frame is None:
            ret, frame = cap.read()
            if not ret:
                break
            last_frame = frame.copy()
        else:
            frame = last_frame.copy()

        # sync: get current frame index
        current_idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1
        current_idx = max(current_idx, 0)

        # draw reference dot if provided
        annotated = frame.copy()
        if current_ref is not None:
            cv2.circle(annotated, current_ref, 5, (255, 0, 0), -1)

        # draw ellipse if valid data exists
        mask = df_current[frame_col] == current_idx
        if mask.any():
            row = df_current[mask].iloc[0]
            cx, cy = row['center_x'], row['center_y']
            if not (pd.isna(cx) or pd.isna(cy)):
                x = int(round(cx))
                y = int(round(cy))
                w = int(row['width'])
                h = int(row['height'])
                phi = float(row['phi'])
                cv2.ellipse(annotated, (x, y), (w, h), phi, 0, 360, (0, 255, 0), 2)

        # final vertical flip for display
        disp = cv2.flip(annotated, 0)
        cv2.imshow('Frame', disp)
        cv2.imshow('Controls', controls_img)

        if cv2.waitKey(30) & 0xFF == 27:  # ESC to exit
            break

    cap.release()
    cv2.destroyAllWindows()


def interactive_eye_rotation_checker(block, eye):
    """
    Interactive tool to visualize how a rotation matrix would affect your
    un-rotated eye data and frame, by drawing the raw ellipse first and then
    warping the entire annotated frame.

    Buttons:
      • Play              : start auto-play
      • Pause             : stop auto-play
      • Original rotation : warp with block.<eye>_rotation_matrix
      • Reversed rotation : warp with inverse matrix
      • Quit              : exit

    Workflow per frame:
      1. Read raw frame (no flips).
      2. Draw ellipse at raw (center_x, center_y) with raw φ from DataFrame.
      3. Warp the *entire* annotated frame by the selected 2×3 matrix.
      4. Vertically flip for display.
    """
    import cv2
    import numpy as np
    import pandas as pd

    # 1) Pick eye‐specific data & matrices
    if eye.lower() == 'left':
        df = block.left_eye_data.copy()
        R_orig = np.array(block.left_rotation_matrix, dtype=np.float32)
        video = block.le_videos[0]
    else:
        df = block.right_eye_data.copy()
        R_orig = np.array(block.right_rotation_matrix, dtype=np.float32)
        video = block.re_videos[0]

    # Compute inverse rotation matrix
    R_rev = cv2.invertAffineTransform(R_orig)

    # Open video
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {eye} video: {video}")
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Determine which column holds frame indices
    frame_col = 'eye_frame' if 'eye_frame' in df.columns else 'frame'

    # 2) Define buttons and layout
    buttons = {
        'Play': ((10, 10), (180, 60)),
        'Pause': ((10, 80), (180, 130)),
        'Original rotation': ((10, 150), (180, 200)),
        'Reversed rotation': ((10, 220), (180, 270)),
        'Quit': ((10, 290), (180, 340)),
    }
    ctrl_h, ctrl_w = 360, 200

    def draw_controls():
        img = np.zeros((ctrl_h, ctrl_w, 3), dtype=np.uint8)
        for name, ((x1, y1), (x2, y2)) in buttons.items():
            cv2.rectangle(img, (x1, y1), (x2, y2), (50, 50, 50), -1)
            cv2.rectangle(img, (x1, y1), (x2, y2), (200, 200, 200), 2)
            cv2.putText(img, name, (x1 + 5, y1 + 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 2, cv2.LINE_AA)
        return img

    controls_img = draw_controls()
    cv2.namedWindow('Controls', cv2.WINDOW_NORMAL)

    # 3) Interaction state
    running = True
    playing = False
    use_matrix = R_orig  # start with original rotation

    def on_mouse(event, x, y, flags, param):
        nonlocal running, playing, use_matrix
        if event != cv2.EVENT_LBUTTONDOWN:
            return
        for name, ((x1, y1), (x2, y2)) in buttons.items():
            if x1 <= x <= x2 and y1 <= y <= y2:
                if name == 'Play':
                    playing = True
                elif name == 'Pause':
                    playing = False
                elif name == 'Original rotation':
                    use_matrix = R_orig
                elif name == 'Reversed rotation':
                    use_matrix = R_rev
                elif name == 'Quit':
                    running = False
                break

    cv2.setMouseCallback('Controls', on_mouse)

    # 4) Playback loop
    last_frame = None
    while running:
        # Advance if playing
        if playing or last_frame is None:
            ret, frame = cap.read()
            if not ret:
                break
            last_frame = frame.copy()
        else:
            frame = last_frame.copy()

        # Sync: current frame index
        idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1
        if idx < 0: idx = 0

        # 5) Draw raw ellipse on native frame
        annotated = frame.copy()
        mask = df[frame_col] == idx
        if mask.any():
            row = df[mask].iloc[0]
            cx, cy = row['center_x'], row['center_y']
            if not (pd.isna(cx) or pd.isna(cy)):
                x = int(round(cx))
                y = int(round(cy))
                w = int(row['width'])
                h = int(row['height'])
                phi = float(row['phi'])
                cv2.ellipse(annotated, (x, y), (w, h), phi, 0, 360, (0, 255, 0), 2)

        # 6) Warp the entire annotated frame
        warped = cv2.warpAffine(
            annotated,
            use_matrix,
            (W, H),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=(0, 0, 0)
        )

        # 7) Final vertical flip for display
        disp = cv2.flip(warped, 0)
        cv2.imshow('Frame', disp)
        cv2.imshow('Controls', controls_img)

        # 8) Exit on ESC
        if cv2.waitKey(30) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()


def negate_eye_rotation(block, eye):
    """
    Replace block.<eye>_rotation_matrix with its inverse (negated rotation)
    and update block.<eye>_rotation_angle to -angle (mod 360).

    Parameters
    ----------
    block : your BlockSync instance
    eye : str
        'left' or 'right'
    """
    if eye.lower() == 'left':
        R_old = np.array(block.left_rotation_matrix, dtype=np.float32)
        ang_old = float(block.left_rotation_angle)
        invR = cv2.invertAffineTransform(R_old)
        block.left_rotation_matrix = invR
        block.left_rotation_angle = (-ang_old) % 360.0
        print(f"Left rotation matrix negated; angle set to {block.left_rotation_angle:.2f}°")
    elif eye.lower() == 'right':
        R_old = np.array(block.right_rotation_matrix, dtype=np.float32)
        ang_old = float(block.right_rotation_angle)
        invR = cv2.invertAffineTransform(R_old)
        block.right_rotation_matrix = invR
        block.right_rotation_angle = (-ang_old) % 360.0
        print(f"Right rotation matrix negated; angle set to {block.right_rotation_angle:.2f}°")
    else:
        raise ValueError("eye must be 'left' or 'right'")


def export_corrected_eye_data(block):
    """
    Overwrite the eye‐data CSVs and rotation‐params pickle so that
    load_eye_data_2d_w_rotation_matrix(block) will load the current,
    corrected attributes from disk.

    Writes:
      • block.analysis_path/'left_eye_data.csv'
      • block.analysis_path/'right_eye_data.csv'
      • block.analysis_path/'rotate_eye_data_params.pkl'
    """
    # Ensure the analysis_path exists
    analysis_path = Path(block.analysis_path)
    analysis_path.mkdir(parents=True, exist_ok=True)

    # 1) Write the DataFrames
    block.left_eye_data.to_csv(analysis_path / 'left_eye_data.csv', index=True)
    block.right_eye_data.to_csv(analysis_path / 'right_eye_data.csv', index=True)

    # 2) Build and write the rotation‐params pickle
    rot_dict = {
        'left_rotation_matrix': block.left_rotation_matrix,
        'left_rotation_angle': block.left_rotation_angle,
        'right_rotation_matrix': block.right_rotation_matrix,
        'right_rotation_angle': block.right_rotation_angle
    }
    with open(analysis_path / 'rotate_eye_data_params.pkl', 'wb') as f:
        pickle.dump(rot_dict, f)

    print(f"Exported corrected eye data and rotation params to {analysis_path}")




In [3]:
def create_block_collections(animals, block_lists, experiment_path, bad_blocks=None):
    """
    Create block collections and a block dictionary from multiple animals and their respective block lists.

    Parameters:
    - animals: list of str, names of the animals.
    - block_lists: list of lists of int, block numbers corresponding to each animal.
    - experiment_path: pathlib.Path, path to the experiment directory.
    - bad_blocks: list of int, blocks to exclude. Default is an empty list.

    Returns:
    - block_collection: list of BlockSync objects for all specified blocks.
    - block_dict: dictionary where keys are block numbers as strings and values are BlockSync objects.
    """
    # uf is already imported at the top of the notebook

    if bad_blocks is None:
        bad_blocks = []

    block_collection = []
    block_dict = {}

    for animal, blocks in zip(animals, block_lists):
        # Generate blocks for the current animal
        current_blocks = uf.block_generator(
            block_numbers=blocks,
            experiment_path=experiment_path,
            animal=animal,
            bad_blocks=bad_blocks
        )
        # Add to collection and dictionary
        block_collection.extend(current_blocks)
        for b in current_blocks:
            block_dict[f"{animal}_block_{b.block_num}"] = b

    return block_collection, block_dict

# Configure your experiment paths and blocks here
animals = ["PV_106"]
block_lists = [[15]]  # Block numbers for each animal
experiment_path = Path(r"D:\sample_data_for_eye_repo")
bad_blocks = []  # Blocks to skip

block_collection, block_dict = create_block_collections(
    animals=animals,
    block_lists=block_lists,
    experiment_path=experiment_path,
    bad_blocks=bad_blocks
)

instantiated block number 015 at Path: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015, new OE version
Found the sample rate for block 015 in the xml file, it is 20000 Hz
created the .oe_rec attribute as an open ephys recording obj with get_data functionality
retrieving zertoh sample number for block 015
got it!


In [4]:
# Load eye data from synchronization pipeline output
# Note: This assumes you've already run the synchronization pipeline
for block in block_collection:
    block.handle_eye_videos()
    try:
        load_eye_data(block)
    except FileNotFoundError:
        print(f'Warning: Eye data not found for block {block.block_num}. Run synchronization pipeline first.')
        continue

handling eye video files
converting videos...
converting files: ['D:\\sample_data_for_eye_repo\\PV_106\\2025_09_04\\block_015\\eye_videos\\LE\\imu_trial4_prey\\imu_trial4_prey.h264', 'D:\\sample_data_for_eye_repo\\PV_106\\2025_09_04\\block_015\\eye_videos\\RE\\imu_trial4_prey\\imu_trial4_prey.h264'] 
 avoiding conversion on files: ['D:\\sample_data_for_eye_repo\\PV_106\\2025_09_04\\block_015\\eye_videos\\LE\\imu_trial4_prey\\imu_trial4_prey_LE.mp4', 'D:\\sample_data_for_eye_repo\\PV_106\\2025_09_04\\block_015\\eye_videos\\RE\\imu_trial4_prey\\imu_trial4_prey.mp4']
The file D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015\eye_videos\RE\imu_trial4_prey\imu_trial4_prey.mp4 already exists, no conversion necessary
Validating videos...
The video named imu_trial4_prey_LE.mp4 has reported 19800 frames and has 19800 frames, it has dropped 0 frames
The video named imu_trial4_prey.mp4 has reported 19701 frames and has 19701 frames, it has dropped 0 frames
Loaded eye data for block 015


In [5]:
block_dict.keys()

dict_keys(['PV_106_block_015'])

In [6]:
# Set up a block for verification
# Update the key to match your animal and block number
block = block_dict['PV_106_block_015']  # Update this to match your data

In [7]:
# Get Kerr reference points if they exist (optional)
# These are used for Kerr degree conversion in downstream analysis
try:
    r_ref = tuple([int(block.kerr_ref_r_x), int(block.kerr_ref_r_y)])
    l_ref = tuple([int(block.kerr_ref_l_x), int(block.kerr_ref_l_y)])
    print(f'Right eye reference: {r_ref}')
    print(f'Left eye reference: {l_ref}')
except AttributeError:
    print('Kerr reference points not set. You can set them using the interactive tool.')
    r_ref = None
    l_ref = None

Kerr reference points not set. You can set them using the interactive tool.


In [7]:
# with kerr reference pick
def interactive_eye_data_corrector_synced(block, eye, ref_point_xy=None):
    """
    Interactive synchronized video + ellipse editor with Play/Pause, correction, Save,
    Flip-Dot, and Skip-forward/backward (1 minute) buttons.

    NEW FEATURE:
    - Click anywhere on the 'Frame' window to set/update the reference point.
    - Press 'Save' to also update block.kerr_ref_<l/r>_<x/y> with the picked reference.

    Parameters
    ----------
    block : BlockSync
        Your BlockSync instance with loaded eye_data.
    eye : str
        'left' or 'right'
    ref_point_xy : tuple[int,int] or None
        If provided, a (x,y) coordinate in raw frame space to draw as a blue dot on every frame.
    """
    import cv2
    import numpy as np
    import pandas as pd

    # 1) select data & video
    eye_lc = eye.lower()
    if eye_lc == 'left':
        df_orig = block.left_eye_data.copy()
        video = block.le_videos[0]
    elif eye_lc == 'right':
        df_orig = block.right_eye_data.copy()
        video = block.re_videos[0]
    else:
        raise ValueError("eye must be 'left' or 'right'")

    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {eye} video: {video}")

    W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    N  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    skip_frames = int(fps * 60)  # skip 1 minute

    # 2) prepare DataFrame & frame index column
    df_current = df_orig.copy()
    frame_col = 'eye_frame' if 'eye_frame' in df_current.columns else 'frame'

    # 3) define buttons & layout (removed rotation-related buttons)
    buttons = {
        'Play': ((10, 10), (180, 60)),
        'Pause': ((10, 80), (180, 130)),
        'X-flip': ((10, 150), (180, 200)),
        'Phi+90': ((10, 220), (180, 270)),
        'FlipX-only': ((10, 290), (180, 340)),
        'Flip Dot': ((10, 360), (180, 410)),
        'Bwd': ((10, 430), (180, 480)),
        'Fwd': ((10, 490), (180, 540)),
        'Save': ((10, 550), (180, 600)),
        'Quit': ((10, 610), (180, 660)),
    }
    ctrl_h, ctrl_w = 680, 200

    def draw_controls():
        img = np.zeros((ctrl_h, ctrl_w, 3), dtype=np.uint8)
        for name, ((x1, y1), (x2, y2)) in buttons.items():
            cv2.rectangle(img, (x1, y1), (x2, y2), (50, 50, 50), -1)
            cv2.rectangle(img, (x1, y1), (x2, y2), (200, 200, 200), 2)
            cv2.putText(img, name, (x1 + 5, y1 + 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2, cv2.LINE_AA)
        return img

    controls_img = draw_controls()
    cv2.namedWindow('Controls', cv2.WINDOW_NORMAL)
    cv2.namedWindow('Frame', cv2.WINDOW_NORMAL)  # ensure we can bind a callback

    # 4) interaction state
    running   = True
    playing   = False
    current_ref = ref_point_xy  # raw-frame coordinates (before final vertical flip)
    last_frame = None

    # --------- Mouse callbacks ----------
    # Controls window: button clicks
    def on_mouse_controls(event, x, y, flags, param):
        nonlocal df_current, running, playing, current_ref, last_frame
        if event != cv2.EVENT_LBUTTONDOWN:
            return
        for name, ((x1, y1), (x2, y2)) in buttons.items():
            if x1 <= x <= x2 and y1 <= y <= y2:
                if name == 'Play':
                    playing = True
                elif name == 'Pause':
                    playing = False
                elif name == 'X-flip':
                    df_current = horizontal_flip_eye_data(df_current, W)
                elif name == 'Phi+90':
                    df_current = rotate_phi_only(df_current)
                elif name == 'FlipX-only':
                    df_current = flip_x_only(df_current, W)
                elif name == 'Flip Dot' and current_ref is not None:
                    x0, y0 = current_ref
                    current_ref = (W - x0, y0)
                elif name == 'Bwd':
                    idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1
                    new_idx = max(idx - skip_frames, 0)
                    cap.set(cv2.CAP_PROP_POS_FRAMES, new_idx)
                    last_frame = None
                elif name == 'Fwd':
                    idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1
                    new_idx = min(idx + skip_frames, N - 1)
                    cap.set(cv2.CAP_PROP_POS_FRAMES, new_idx)
                    last_frame = None
                elif name == 'Save':
                    # Save DataFrame edits
                    if eye_lc == 'left':
                        block.left_eye_data = df_current.copy()
                    else:
                        block.right_eye_data = df_current.copy()
                    # Save reference point to block attributes if available
                    if current_ref is not None:
                        rx = int(round(current_ref[0]))
                        ry = int(round(current_ref[1]))
                        if eye_lc == 'left':
                            block.kerr_ref_l_x = rx
                            block.kerr_ref_l_y = ry
                            print(f"Saved left-eye reference to block: ({rx}, {ry})")
                        else:
                            block.kerr_ref_r_x = rx
                            block.kerr_ref_r_y = ry
                            print(f"Saved right-eye reference to block: ({rx}, {ry})")
                    print(f"{eye.capitalize()} eye data saved.")
                elif name == 'Quit':
                    running = False
                break

    # Frame window: click to set reference
    # NOTE: The displayed frame is vertically flipped for viewing. Map click -> raw frame coords.
    def on_mouse_frame(event, x, y, flags, param):
        nonlocal current_ref
        if event != cv2.EVENT_LBUTTONDOWN:
            return
        # y in 'Frame' is after a vertical flip; convert back to raw-frame coordinates:
        y_raw = H - 1 - y
        x_raw = x
        current_ref = (int(x_raw), int(y_raw))
        # Provide visual/console feedback:
        print(f"Picked reference (raw coords): ({current_ref[0]}, {current_ref[1]})")

    cv2.setMouseCallback('Controls', on_mouse_controls)
    cv2.setMouseCallback('Frame', on_mouse_frame)

    # 6) play/pause loop
    while running:
        if playing or last_frame is None:
            ret, frame = cap.read()
            if not ret:
                break
            last_frame = frame.copy()
        else:
            frame = last_frame.copy()

        # sync: get current frame index
        current_idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES)) - 1
        current_idx = max(current_idx, 0)

        # draw reference dot if provided (raw-frame coords)
        annotated = frame.copy()
        if current_ref is not None:
            cv2.circle(annotated, (int(current_ref[0]), int(current_ref[1])), 5, (255, 0, 0), -1)

        # draw ellipse if valid data exists
        mask = df_current[frame_col] == current_idx
        if mask.any():
            row = df_current[mask].iloc[0]
            cx, cy = row['center_x'], row['center_y']
            if not (pd.isna(cx) or pd.isna(cy)):
                x = int(round(cx))
                y = int(round(cy))
                w = int(row.get('width', 0))
                h = int(row.get('height', 0))
                phi = float(row.get('phi', 0.0))
                # Guard widths/heights
                w = max(w, 1); h = max(h, 1)
                cv2.ellipse(annotated, (x, y), (w, h), phi, 0, 360, (0, 255, 0), 2)

        # final vertical flip for display (maintains your y-positive-up convention)
        disp = cv2.flip(annotated, 0)
        cv2.imshow('Frame', disp)
        cv2.imshow('Controls', controls_img)

        if cv2.waitKey(30) & 0xFF == 27:  # ESC to exit
            break

    cap.release()
    cv2.destroyAllWindows()

import pandas as pd
from pathlib import Path
import numpy as np

def export_current_kerr_refs(block, filename: str = "self_kerr_refs.csv") -> Path:
    """
    Save the current block's Kerr reference coordinates to a small CSV in the analysis folder.

    Writes a single-row CSV with columns:
        kerr_ref_r_x, kerr_ref_r_y, kerr_ref_l_x, kerr_ref_l_y

    Returns
    -------
    Path
        The path to the written CSV.
    """
    analysis_path = Path(block.analysis_path)
    analysis_path.mkdir(parents=True, exist_ok=True)

    # Pull attributes if present; otherwise write NaN so the schema stays consistent
    vals = {
        "kerr_ref_r_x": getattr(block, "kerr_ref_r_x", np.nan),
        "kerr_ref_r_y": getattr(block, "kerr_ref_r_y", np.nan),
        "kerr_ref_l_x": getattr(block, "kerr_ref_l_x", np.nan),
        "kerr_ref_l_y": getattr(block, "kerr_ref_l_y", np.nan),
    }

    out_path = analysis_path / filename
    pd.DataFrame([vals]).to_csv(out_path, index=False)
    print(f"Kerr refs exported to: {out_path}")
    return out_path


def load_self_kerr_refs(block, filename: str = "self_kerr_refs.csv") -> bool:
    """
    Load Kerr reference coordinates from the analysis folder CSV and set them on `block`.

    Reads a single-row CSV with columns:
        kerr_ref_r_x, kerr_ref_r_y, kerr_ref_l_x, kerr_ref_l_y

    Returns
    -------
    bool
        True if refs were loaded and applied, False if the file was missing or empty.
    """
    path = Path(block.analysis_path) / filename
    if not path.exists():
        print(f"No Kerr refs file found at: {path}")
        return False

    df = pd.read_csv(path)
    if df.empty:
        print(f"Kerr refs file is empty: {path}")
        return False

    row = df.iloc[0]

    # Helper to safely set attribute if value is finite
    def _set_attr(name):
        if name in row and pd.notna(row[name]):
            try:
                setattr(block, name, int(round(float(row[name]))))
            except (ValueError, TypeError):
                # keep existing value if conversion fails
                pass

    for col in ("kerr_ref_r_x", "kerr_ref_r_y", "kerr_ref_l_x", "kerr_ref_l_y"):
        _set_attr(col)

    print(f"Kerr refs loaded from: {path}")
    return True


In [8]:
# use the interactive tool to get a properly aligned data on a native frame (this is y-flipped after plotting and maintains the y-positive = up convention
# save before exiting the tool!!
interactive_eye_data_corrector_synced(block, eye='left')

Picked reference (raw coords): (388, 315)
Picked reference (raw coords): (394, 293)
Picked reference (raw coords): (414, 284)
Picked reference (raw coords): (414, 299)
Picked reference (raw coords): (410, 288)
Picked reference (raw coords): (400, 268)
Saved left-eye reference to block: (400, 268)
Left eye data saved.


In [9]:
# run for right eye
interactive_eye_data_corrector_synced(block, eye='right')

Picked reference (raw coords): (282, 366)
Picked reference (raw coords): (294, 352)
Saved right-eye reference to block: (294, 352)
Right eye data saved.


In [10]:
# Export corrected eye data
def export_corrected_eye_data(block):
    """
    Export the corrected eye data dataframes to CSV files.
    No rotation matrices are exported as rotation is no longer used.
    """
    block.right_eye_data.to_csv(block.analysis_path / 'right_eye_data.csv')
    block.left_eye_data.to_csv(block.analysis_path / 'left_eye_data.csv')
    print(f'Exported corrected eye data to {block.analysis_path}')

# Export the corrected data
export_corrected_eye_data(block)

Exported corrected eye data to D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015\analysis


In [11]:
export_current_kerr_refs(block)

Kerr refs exported to: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015\analysis\self_kerr_refs.csv


WindowsPath('D:/sample_data_for_eye_repo/PV_106/2025_09_04/block_015/analysis/self_kerr_refs.csv')